<a href="https://colab.research.google.com/github/aryanshenoy02/Banking-Fraud-Detection/blob/main/notebooks/isolation-forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

# uploaded is a dict like {'whatever_name.csv': b'...bytes...'}
# next(iter(...)) grabs the first (and only) key, i.e. the filename
filename = next(iter(uploaded))

try:
    # Skipping the first 3 rows of metadata to start directly from the table headers
    df = pd.read_csv(filename, skiprows=3)
    print(f"Successfully loaded '{filename}' by skipping the metadata rows:")
    display(df.head(10))
except Exception as e:
    print(f"Error reading CSV: {e}")

In [ ]:
combined = df['Date'].astype(str) + ' ' + df['Time'].astype(str)

parsed = pd.to_datetime(
    combined.str.replace(r'\bSept\b', 'Sep', regex=True),
    format='%b %d, %Y %I:%M %p',
    errors='coerce'
)

invalid_rows = df[parsed.isna()]

display(invalid_rows)
df = df[parsed.notna()].copy()

df['Datetime'] = parsed[parsed.notna()]

In [ ]:
df['Date'] = df['Date'].str.replace(
    r'\bSept\b', 'Sep', regex=True
)

df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%b %d, %Y %I:%M %p'
)

In [ ]:
import numpy as np

# 1. Drop rows where all columns are NaN (if any exist at the end of the file)
df = df.dropna(how='all').copy()

# 2. Clean the 'Transaction Details' column
# We remove case-insensitive variations of "Paid to " and "Received from "
df['Transaction Details'] = df['Transaction Details'].str.replace(r'^(Paid to|Received from)\s+', '', case=False, regex=True)

# 3. Create a combined datetime column to sort chronologically and compute differences
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%b %d, %Y %I:%M %p')

# Sort chronologically so we can calculate the time difference forward in time
df = df.sort_values('Datetime').reset_index(drop=True)

# 4. Calculate the time difference between consecutive transactions (in hours)
df['Time_Diff_Hours'] = df['Datetime'].diff().dt.total_seconds() / 3600.0

# Display the cleaned and processed DataFrame
display(df[['Date', 'Time', 'Transaction Details', 'Transaction Type', 'Amount', 'Time_Diff_Hours']].head(10))

In [ ]:
# sort to datatime
df = df.sort_values('Datetime').reset_index(drop=True)

# Calculate cumulative transaction count per counterparty (excluding current transaction)
df['Counterparty_Transaction_Count'] = df.groupby('Transaction Details').cumcount()

# Display the results emphasizing the new column
display(df[['Datetime', 'Transaction Details', 'Counterparty_Transaction_Count']].head(20))

In [ ]:
import numpy as np

df['Log_Amount'] = np.log1p(df['Amount'])
df = df.drop(columns = ['Transaction ID', 'UTR', 'Date', 'Time'])
df.head()

In [ ]:
df = df.sort_values('Datetime').reset_index(drop=True)

# last 24 hours (time-based), current row excluded
past24 = df.set_index('Datetime')['Log_Amount'].rolling('24h', closed='left').mean()
df['dev_24h'] = (df['Log_Amount'] - past24.values).fillna(0).clip(-2, 2)

# last 6 transactions (count-based), current row excluded via shift
past6 = df['Log_Amount'].shift().rolling(6, min_periods=1).mean()
df['dev_last6'] = (df['Log_Amount'] - past6).fillna(0).clip(-2, 2)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

df['hour'] = df['Datetime'].dt.hour
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dayofweek'] = df['Datetime'].dt.dayofweek
df['is_new_counterparty'] = (df['Counterparty_Transaction_Count'] == 0).astype(int)
df['is_debit'] = (df['Transaction Type'] == 'DEBIT').astype(int)
#df = pd.get_dummies(df, columns=['Credit/debit instrument'], dtype=int)

imp_features = ['Log_Amount']
core_features = ['Time_Diff_Hours', 'is_new_counterparty',
                 'hour_sin', 'hour_cos', 'dayofweek', 'is_debit']
extra_features = ['dev_24h', 'dev_last6']

# 2. Split by date
cut = int(len(df) * 0.75)
train = df.iloc[:cut].copy()
test = df.iloc[cut:].copy()

#for the 1st value since its NaN
median_gap = train['Time_Diff_Hours'].median()
train['Time_Diff_Hours'] = train['Time_Diff_Hours'].fillna(median_gap)
test['Time_Diff_Hours'] = test['Time_Diff_Hours'].fillna(median_gap)

def make_scorer(cols, weight):
    m = IsolationForest(n_estimators=200, random_state=42).fit(train[cols])
    raw_train = -m.decision_function(train[cols])
    lo, hi = raw_train.min(), raw_train.max()
    def score(d):
        raw = -m.decision_function(d[cols])
        return weight * np.clip((raw - lo) / (hi - lo), 0, None)
    return score

imp_score = make_scorer(imp_features, weight=1.2)
core_score = make_scorer(core_features, weight=1.0)
extra_score = make_scorer(extra_features, weight=0.3)

train['anomaly'] = core_score(train) + extra_score(train) + imp_score(train)
test['anomaly'] = core_score(test) + extra_score(test) + imp_score(test)

In [ ]:
# Combine train and test sets to find top overall anomalies, or we can look at df
# Since 'anomaly' was assigned separately to train and test, let's combine them back or use df
df_scored = pd.concat([train, test]).sort_values(by='anomaly', ascending=False)

# Select the requested columns and display the top 10
top_anomalies = df_scored[['Amount', 'anomaly', 'Transaction Details', 'Datetime']].head(10)
display(top_anomalies)